# Cox Analysis Recreation

This notebook rebuilds the Cox tables and forest plots from the staged panel-seq inputs plus one external clinical metadata TSV.

Like the other notebooks in `analysis/`, the repository does not carry raw clinical cleanup. The external clinical file should already be preformatted before running this notebook.


## External Clinical File Contract

The shared external TSV should contain one row per primary tumor `Sample` and the cleaned columns used below.

Required survival and cohort columns:
- `Sample`
- `patient_id`
- `relapse_any`
- `relapse_120m`
- `rfs_time_months`
- `sentinel`

Required clinicopathologic columns:
- `age_at_diagnosis_years` or `age_years`
- `sex`
- `ajcc8_stage` or `stage_cat`
- `tumor_thickness_mm`
- `tumor_thickness_group`
- `ulceration`
- `histology`
- `primary_site`
- `regression`
- `nevus_association`

If your external file uses the earlier Figure 1 names where available, this notebook accepts those as fallbacks.


In [ ]:
from __future__ import annotations

import glob
import gzip
import os
import re
import warnings
from collections import defaultdict
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from lifelines import CoxPHFitter
from scipy.stats import chi2, chi2_contingency, fisher_exact, mannwhitneyu

DEFAULT_NOTEBOOK_DIR = Path("/mnt/myvolume/PanelSeqMelanomaNotebook/analysis")
NOTEBOOK_DIR = Path(os.environ.get("PANELSEQ_NOTEBOOK_DIR", DEFAULT_NOTEBOOK_DIR)).expanduser().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent
PANEL_SEQ_ROOT = Path(os.environ.get("PANELSEQ_DATA_ROOT", "/mnt/myvolume/panel_seq")).expanduser().resolve()

CLINICAL_METADATA_PATH = Path(
    os.environ.get(
        "PANELSEQ_FIG1_CLINICAL_PATH",
        str(PANEL_SEQ_ROOT / "figure_inputs/figure1_clinical_minimal.tsv"),
    )
).expanduser().resolve()
VCF_DIR = PANEL_SEQ_ROOT / "new_bed_analysis/vcfs"
CLINCNV_ROOT = PANEL_SEQ_ROOT / "v4_v5_clincnv/new_clincnv_runs/combined_runs/combined_bed_split_s100_l3_f1"
ONCOKB_DIR = PANEL_SEQ_ROOT / "new_bed_analysis/oncokb/oncokb_results_pass_vaf002_alt5"
MEL_ANNOTATION_PATH = PANEL_SEQ_ROOT / "v4_v5_clincnv/MEL_Annotation.xlsx"
PANEL_BED_PATHS = {
    "v4": PANEL_SEQ_ROOT / "v4_v5_clincnv/ssSC_v4.bed",
    "v5": PANEL_SEQ_ROOT / "v4_v5_clincnv/ssSC_v5.bed",
}

LOSS_GENES_SIX = ["CDKN2A", "CDKN2B", "TP53BP1"]
GAIN_GENES_SIX = ["CDK4", "CD276", "MCL1"]
ONCOCYCLE_FEATURES = [f"{gene} loss" for gene in LOSS_GENES_SIX] + [f"{gene} gain" for gene in GAIN_GENES_SIX]
MAJOR_ONCOGENIC_GENE_FEATURES = [
    "BRAF_NRAS_NF1_any",
    "TERT_oncogenic",
    "BRAF_oncogenic",
    "NRAS_oncogenic",
    "NF1_oncogenic",
    "KIT_oncogenic",
    "CDKN2A_oncogenic",
    "TP53_oncogenic",
    "RAC1_oncogenic",
    "MAP2K1_oncogenic",
    "LRP1B_oncogenic",
    "PTPRT_oncogenic",
    "PTPRD_oncogenic",
    "ARID2_oncogenic",
    "PPP6C_oncogenic",
]
BASES = set("ACGT")
VCF_GLOB = ["**/filtered.*.annotated.vcf", "**/filtered.*.annotated.vcf.gz"]
STRICT_SOMATIC = False
TMB_OVERLAP_PADDING = 100
FGA_INTERVAL_PADDING = 100
NONSYN_CLASSES = {
    "Missense_Mutation",
    "Nonsense_Mutation",
    "Nonstop_Mutation",
    "Splice_Site",
    "Frame_Shift_Ins",
    "Frame_Shift_Del",
    "In_Frame_Ins",
    "In_Frame_Del",
    "Translation_Start_Site",
}
EFFECT_TO_CLASS = {
    "frameshift_variant": "Frame_Shift",
    "inframe_insertion": "In_Frame_Ins",
    "disruptive_inframe_insertion": "In_Frame_Ins",
    "conservative_inframe_insertion": "In_Frame_Ins",
    "inframe_deletion": "In_Frame_Del",
    "disruptive_inframe_deletion": "In_Frame_Del",
    "conservative_inframe_deletion": "In_Frame_Del",
    "missense_variant": "Missense_Mutation",
    "protein_altering_variant": "Missense_Mutation",
    "coding_sequence_variant": "Missense_Mutation",
    "stop_gained": "Nonsense_Mutation",
    "stop_lost": "Nonstop_Mutation",
    "start_lost": "Translation_Start_Site",
    "splice_acceptor_variant": "Splice_Site",
    "splice_donor_variant": "Splice_Site",
    "splice_region_variant": "Splice_Site",
}
SEVERITY_ORDER = [
    "frameshift_variant",
    "stop_gained",
    "stop_lost",
    "splice_acceptor_variant",
    "splice_donor_variant",
    "splice_region_variant",
    "start_lost",
    "inframe_deletion",
    "disruptive_inframe_deletion",
    "conservative_inframe_deletion",
    "inframe_insertion",
    "disruptive_inframe_insertion",
    "conservative_inframe_insertion",
    "missense_variant",
    "protein_altering_variant",
    "coding_sequence_variant",
]
SEVERITY_RANK = {effect: rank for rank, effect in enumerate(SEVERITY_ORDER)}

plt.rcParams.update({
    "figure.dpi": 180,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Liberation Sans",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


REQUIRED_CLINICAL_COLUMNS = [
    "Sample",
    "patient_id",
    "relapse_any",
    "relapse_120m",
    "rfs_time_months",
    "sentinel",
]

CLINICAL_COLUMN_ALIASES = {
    "age_at_diagnosis_years": ["age_at_diagnosis_years", "age_years"],
    "sex": ["sex"],
    "ajcc8_stage": ["ajcc8_stage", "stage_cat"],
    "tumor_thickness_mm": ["tumor_thickness_mm"],
    "tumor_thickness_group": ["tumor_thickness_group"],
    "ulceration": ["ulceration"],
    "histology": ["histology", "histology_cat"],
    "primary_site": ["primary_site"],
    "regression": ["regression"],
    "nevus_association": ["nevus_association"],
}


def glob_count(pattern):
    return len(glob.glob(str(pattern), recursive=True))


def path_count(path_like):
    return int(Path(path_like).exists())


def fmt_p(value):
    if pd.isna(value):
        return ""
    value = float(value)
    if value < 1e-4:
        return f"{value:.2e}"
    return f"{value:.6f}".rstrip("0").rstrip(".")


def fmt_p_plain(value):
    if pd.isna(value):
        return ""
    value = float(value)
    return f"{value:.6g}" if value < 1e-4 else f"{value:.6f}"


def fmt_hr_ci(hr, low, high, digits=2):
    return f"{float(hr):.{digits}f} [{float(low):.{digits}f}-{float(high):.{digits}f}]"


def open_maybe_gzip(path):
    path = str(path)
    return gzip.open(path, "rt") if path.endswith(".gz") else open(path, "rt")


def sample_name_from_path(path):
    base = os.path.basename(str(path))
    if base.startswith("filtered."):
        base = base[len("filtered."):]
    for suffix in (".annotated.vcf.gz", ".annotated.vcf", ".vcf.gz", ".vcf"):
        if base.endswith(suffix):
            return base[:-len(suffix)]
    return base


def normalize_sample_id(value):
    return str(value).split("__")[0].strip()


def canon_chrom(value):
    if value is None:
        return ""
    return re.sub(r"^chr", "", str(value).strip(), flags=re.IGNORECASE)


def parse_info_dict(info):
    parsed = {}
    for item in str(info or "").split(";"):
        if not item:
            continue
        if "=" in item:
            key, value = item.split("=", 1)
            parsed[key] = value
        else:
            parsed[item] = True
    return parsed


def looks_somatic(info, strict=False):
    if not strict:
        return True
    markers = ["SOMATIC", "SOMATIC=1", "SS=2", "SSTATUS=Somatic", "TYPE=somatic", "SOMATIC;"]
    info_text = str(info or "")
    return any(marker in info_text for marker in markers)


def is_snv(ref, alt):
    return len(ref) == 1 and len(alt) == 1 and ref in BASES and alt in BASES


def get_vaf_dp(format_keys, sample_values, alt_index, info_dict=None):
    if not format_keys or not sample_values:
        vaf = None
        if info_dict and "AF" in info_dict:
            af_values = str(info_dict["AF"]).split(",")
            if alt_index < len(af_values):
                try:
                    vaf = float(af_values[alt_index])
                except Exception:
                    vaf = None
        return vaf, None

    fmt = dict(zip(format_keys, sample_values))
    dp = None
    if "DP" in fmt and fmt["DP"] not in (None, ".", ""):
        try:
            dp = int(float(fmt["DP"]))
        except Exception:
            dp = None

    vaf = None
    if "AF" in fmt and fmt["AF"] not in (None, ".", ""):
        af_values = str(fmt["AF"]).split(",")
        if alt_index < len(af_values):
            try:
                vaf = float(af_values[alt_index])
            except Exception:
                vaf = None

    if vaf is None and "AD" in fmt and fmt["AD"] not in (None, ".", ""):
        try:
            ad_values = [int(x) for x in str(fmt["AD"]).split(",")]
            if len(ad_values) >= 2 and (alt_index + 1) < len(ad_values):
                ref_count = ad_values[0]
                alt_count = ad_values[alt_index + 1]
                denom = ref_count + alt_count
                vaf = (alt_count / denom) if denom > 0 else 0.0
                if dp is None:
                    dp = int(sum(ad_values))
        except Exception:
            pass

    if vaf is None and info_dict and "AF" in info_dict:
        af_values = str(info_dict["AF"]).split(",")
        if alt_index < len(af_values):
            try:
                vaf = float(af_values[alt_index])
            except Exception:
                vaf = None
    return vaf, dp


def split_genes(value):
    if pd.isna(value) or value == "":
        return set()
    return {gene.strip().upper() for gene in str(value).split(",") if gene.strip()}


def clean_gene_name(value):
    gene = str(value).strip().upper()
    if not gene or gene in {"0", ".", "NA"} or re.search(r"[A-Z]", gene) is None:
        return ""
    return gene


def detect_oncokb_variant_columns(df):
    required = ["Chromosome", "Start_Position", "Reference_Allele", "Tumor_Seq_Allele2"]
    if all(column in df.columns for column in required):
        return ("Chromosome", "Start_Position", "Reference_Allele", "Tumor_Seq_Allele2")
    return None


def rowwise_key_membership(df, chrom_col, pos_col, ref_col, alt_col, vset):
    tmp = df[[chrom_col, pos_col, ref_col, alt_col]].copy()
    tmp["chrom"] = tmp[chrom_col].map(canon_chrom)
    tmp["pos"] = pd.to_numeric(tmp[pos_col], errors="coerce")
    tmp["ref"] = tmp[ref_col].astype(str)
    tmp["alt"] = tmp[alt_col].astype(str)

    def in_vcf(row):
        if pd.isna(row["pos"]):
            return False
        alts = [alt.strip() for alt in str(row["alt"]).split(",") if alt.strip() and alt.strip() != "."]
        return any(f"{row['chrom']}:{int(row['pos'])}:{row['ref']}:{alt}" in vset for alt in alts)

    return tmp.apply(in_vcf, axis=1)


def oncokb_sid_from_file(path):
    base = os.path.basename(str(path))
    stem = os.path.splitext(base)[0]
    if stem.endswith(".oncokb"):
        stem = stem[:-len(".oncokb")]
    if stem.startswith("filtered."):
        stem = stem[len("filtered."):]
    return normalize_sample_id(stem)


def extract_genes_from_label_list(label_str):
    if not isinstance(label_str, str) or not label_str.strip():
        return ""
    genes = []
    for item in [x.strip() for x in label_str.split(",") if x.strip()]:
        match = re.match(r"^([A-Za-z0-9]+)", item)
        if match:
            genes.append(match.group(1).upper())
    return ",".join(sorted(set(genes)))


def count_items(value):
    if not isinstance(value, str) or not value.strip():
        return 0
    return sum(1 for item in value.split(",") if item.strip())


def merge_gene_lists(series):
    merged = set()
    for values in series:
        if isinstance(values, list):
            merged.update(values)
    return sorted(merged)


def merge_max_dicts(series):
    merged = {}
    for item in series:
        if not isinstance(item, dict):
            continue
        for gene, value in item.items():
            merged[gene] = max(value, merged.get(gene, float("-inf")))
    return merged


def _state_to_bucket(state):
    state = str(state).strip().upper()
    if state == "AMP":
        return "amp"
    if state == "GAIN":
        return "gained"
    if state == "DEL":
        return "lost"
    if state == "LOH":
        return "loh"
    return None


def _coerce_int_like(value):
    try:
        return int(float(value))
    except Exception:
        return None


def _event_chrom(event):
    for key in event.keys():
        if key.lower() in {"chr", "chrom", "chromosome"}:
            raw = str(event[key]).strip()
            if not raw:
                return None
            return re.sub(r"^chr", "", raw, flags=re.IGNORECASE).upper()
    return None


def _classify_del(event):
    for major_key, minor_key in [("major_CN_allele", "minor_CN_allele"), ("major_CN_allele2", "minor_CN_allele2")]:
        major = _coerce_int_like(event.get(major_key))
        minor = _coerce_int_like(event.get(minor_key))
        if major == 0 and minor == 0:
            return "HOMDEL"
    return "HETDEL"


def _read_clincnv_table(path):
    rows = []
    header_cols = None
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("##"):
                continue
            if line.startswith("#"):
                header_cols = [column.strip() for column in line[1:].rstrip("\n").split("\t")]
                continue
            if not line.strip() or header_cols is None:
                continue
            values = [value.strip() for value in line.rstrip("\n").split("\t")]
            if len(values) < len(header_cols):
                values += [""] * (len(header_cols) - len(values))
            rows.append(dict(zip(header_cols, values[:len(header_cols)])))
    return rows


def sample_from_clincnv_path(path):
    base = os.path.basename(str(path))
    patterns = (
        r"(?:Annotated_CNA_CNAs?|Annotated_ClinCNV)_(.+?)\.txt$",
        r".+?_(QHPRG[^_]+)_Annotated_ClinCNV\.txt$",
    )
    for pattern in patterns:
        match = re.match(pattern, base)
        if match:
            return match.group(1)
    return os.path.splitext(base)[0]


def _summarize_clincnv_sample(path):
    sample = sample_from_clincnv_path(path).split("-")[0]
    genes_amp, genes_gained = set(), set()
    genes_het_del, genes_hom_del, genes_loh = set(), set(), set()
    amp_like_cn_by_gene = {}

    for event in _read_clincnv_table(path):
        bucket = _state_to_bucket(event.get("state"))
        if bucket is None:
            continue
        chrom = _event_chrom(event)
        genes = [gene.strip().upper() for gene in str(event.get("genes", "")).split(",") if gene.strip()]
        total_cn = None
        try:
            total_cn = float(event.get("tumor_CN_change"))
        except Exception:
            total_cn = None

        if bucket in {"amp", "gained"}:
            target = genes_amp if bucket == "amp" else genes_gained
            target.update(genes)
            for gene in genes:
                if total_cn is not None:
                    amp_like_cn_by_gene[gene] = max(total_cn, amp_like_cn_by_gene.get(gene, float("-inf")))
        elif bucket == "lost":
            if chrom == "Y":
                continue
            if _classify_del(event) == "HOMDEL":
                genes_hom_del.update(genes)
            else:
                genes_het_del.update(genes)
        elif bucket == "loh":
            if chrom in {"X", "Y"}:
                continue
            genes_loh.update(genes)

    amp_like_cn_by_gene = {gene: cn for gene, cn in amp_like_cn_by_gene.items() if cn != float("-inf")}
    return {
        "sample": sample,
        "genes_amp": sorted(genes_amp),
        "genes_gained": sorted(genes_gained),
        "genes_het_del": sorted(genes_het_del),
        "genes_hom_del": sorted(genes_hom_del),
        "genes_loh": sorted(genes_loh),
        "amp_like_cn_by_gene": amp_like_cn_by_gene,
    }


def iter_clincnv_paths(root_dir):
    root_dir = Path(root_dir)
    patterns = [
        "**/Annotated_ClinCNV_*.txt",
        "**/Annotated_CNA_CNAs_*.txt",
        "**/CNAs_*.txt",
        "**/*_Annotated_ClinCNV.txt",
    ]
    seen = set()
    for pattern in patterns:
        for path in sorted(root_dir.glob(pattern)):
            if path.is_file() and path not in seen and re.search(r"QHPRG", path.name, re.IGNORECASE):
                seen.add(path)
                yield path


def build_clincnv_summary(root_dir):
    records = [_summarize_clincnv_sample(path) for path in iter_clincnv_paths(root_dir)]
    raw_df = pd.DataFrame(records)
    if raw_df.empty:
        return pd.DataFrame()
    grouped = raw_df.groupby("sample", as_index=True).agg({
        "genes_amp": merge_gene_lists,
        "genes_gained": merge_gene_lists,
        "genes_het_del": merge_gene_lists,
        "genes_hom_del": merge_gene_lists,
        "genes_loh": merge_gene_lists,
        "amp_like_cn_by_gene": merge_max_dicts,
    })
    grouped["genes_amp_str"] = grouped["genes_amp"].apply(lambda values: ", ".join(values))
    grouped["genes_gained_str"] = grouped["genes_gained"].apply(lambda values: ", ".join(values))
    grouped["genes_HETDEL_str"] = grouped["genes_het_del"].apply(lambda values: ", ".join(values))
    grouped["genes_HOMDEL_str"] = grouped["genes_hom_del"].apply(lambda values: ", ".join(values))
    grouped["genes_LOH_str"] = grouped["genes_loh"].apply(lambda values: ", ".join(values))
    return grouped


def build_oncokb_summary(oncokb_dir, vcf_dir, include_samples=None):
    vcf_paths = set()
    for pattern in VCF_GLOB:
        vcf_paths |= set(glob.glob(str(Path(vcf_dir) / pattern), recursive=True))

    sample_to_vcf = {}
    for path in sorted(vcf_paths):
        sid = normalize_sample_id(sample_name_from_path(path))
        sample_to_vcf.setdefault(sid, path)

    vcf_keyset = {}
    for sid, vcf_path in sample_to_vcf.items():
        keys = set()
        with open_maybe_gzip(vcf_path) as handle:
            for line in handle:
                if not line or line.startswith("#"):
                    continue
                cols = line.rstrip("\n").split("\t")
                if len(cols) < 10:
                    continue
                chrom, pos, _id, ref, alts, qual, flt, info = cols[:8]
                if flt != "PASS" or not looks_somatic(info, strict=STRICT_SOMATIC):
                    continue
                info_dict = parse_info_dict(info)
                format_keys = cols[8].split(":")
                sample_values = cols[9].split(":")
                for alt_index, alt in enumerate(alts.split(",")):
                    vaf, _ = get_vaf_dp(format_keys, sample_values, alt_index, info_dict=info_dict)
                    if vaf is None:
                        continue
                    keys.add(f"{canon_chrom(chrom)}:{int(pos)}:{ref}:{alt}")
        vcf_keyset[sid] = keys

    rows = []
    for path in sorted(Path(oncokb_dir).glob("*.oncokb.tsv")):
        df = pd.read_csv(path, sep="\t", dtype=str).fillna("")
        sid = oncokb_sid_from_file(path)
        vcf_keys = vcf_keyset.get(sid, set())
        variant_cols = detect_oncokb_variant_columns(df)
        filtered_df = df if variant_cols is None else df.loc[rowwise_key_membership(df, *variant_cols, vcf_keys)].copy()

        oncogenic = filtered_df.get("ONCOGENIC", pd.Series("", index=filtered_df.index)).astype(str)
        mutation_effect = filtered_df.get("MUTATION_EFFECT", pd.Series("", index=filtered_df.index)).astype(str)
        hugo_symbol = filtered_df.get("ONCOKB_HUGO_SYMBOL", pd.Series("", index=filtered_df.index)).astype(str)
        protein_change = filtered_df.get("ONCOKB_PROTEIN_CHANGE", pd.Series("", index=filtered_df.index)).astype(str)

        oncogenic_mask = oncogenic.str.contains("Oncogenic", case=False, na=False)
        gof_mask = mutation_effect.str.contains("Gain-of-function", case=False, na=False)
        lof_mask = mutation_effect.str.contains("Loss-of-function", case=False, na=False)
        filtered_df["ONCOKB_HUGO_SYMBOL_CLEAN"] = hugo_symbol.map(clean_gene_name)
        filtered_df["variant_label"] = (
            filtered_df["ONCOKB_HUGO_SYMBOL_CLEAN"].astype(str)
            + " "
            + protein_change.astype(str)
        ).str.strip()
        valid_gene_mask = filtered_df["ONCOKB_HUGO_SYMBOL_CLEAN"].ne("")

        oncogenic_variants = filtered_df.loc[valid_gene_mask & oncogenic_mask, "variant_label"].dropna().unique().tolist()
        oncogenic_genes = filtered_df.loc[valid_gene_mask & oncogenic_mask, "ONCOKB_HUGO_SYMBOL_CLEAN"].dropna().astype(str).unique().tolist()
        gof_variants = filtered_df.loc[valid_gene_mask & oncogenic_mask & gof_mask, "variant_label"].dropna().unique().tolist()
        lof_one = filtered_df.loc[valid_gene_mask & oncogenic_mask & lof_mask, "variant_label"].dropna().unique().tolist()
        lof_counts = filtered_df.loc[valid_gene_mask & oncogenic_mask & lof_mask, "ONCOKB_HUGO_SYMBOL_CLEAN"].value_counts()
        lof_biallelic = lof_counts[lof_counts > 1].index.tolist()

        rows.append({
            "Sample": sid,
            "Oncogenic_Variants": ", ".join(oncogenic_variants) if oncogenic_variants else "",
            "Oncogenic_Genes": ",".join(sorted(set(oncogenic_genes))) if oncogenic_genes else "",
            "GOF_Oncogenic_Variants": ", ".join(gof_variants) if gof_variants else "",
            "LOF_Oncogenic_One_Allele": ", ".join(lof_one) if lof_one else "",
            "LOF_Oncogenic_Both_Alleles": ", ".join(lof_biallelic) if lof_biallelic else "",
        })

    summary_df = pd.DataFrame(rows)
    if summary_df.empty:
        summary_df = pd.DataFrame(columns=[
            "Oncogenic_Variants",
            "Oncogenic_Genes",
            "GOF_Oncogenic_Variants",
            "LOF_Oncogenic_One_Allele",
            "LOF_Oncogenic_Both_Alleles",
        ])
        summary_df.index.name = "Sample"
    else:
        summary_df["GOF_Genes"] = summary_df["GOF_Oncogenic_Variants"].map(extract_genes_from_label_list)
        summary_df["LOF_OneAllele_Genes"] = summary_df["LOF_Oncogenic_One_Allele"].map(extract_genes_from_label_list)
        summary_df["LOF_Biallelic_Genes"] = summary_df["LOF_Oncogenic_Both_Alleles"].map(extract_genes_from_label_list)
        summary_df["Variant_Richness"] = summary_df["Oncogenic_Variants"].map(count_items)
        summary_df = summary_df.sort_values(["Sample", "Variant_Richness"], ascending=[True, False]).drop_duplicates("Sample", keep="first").set_index("Sample")

    if include_samples is not None:
        include_index = pd.Index([normalize_sample_id(sample) for sample in include_samples], name="Sample").drop_duplicates()
        summary_df = summary_df.reindex(include_index)
        for column in [
            "Oncogenic_Variants",
            "Oncogenic_Genes",
            "GOF_Oncogenic_Variants",
            "LOF_Oncogenic_One_Allele",
            "LOF_Oncogenic_Both_Alleles",
            "GOF_Genes",
            "LOF_OneAllele_Genes",
            "LOF_Biallelic_Genes",
        ]:
            if column in summary_df.columns:
                summary_df[column] = summary_df[column].fillna("").astype(str)
        if "Variant_Richness" in summary_df.columns:
            summary_df["Variant_Richness"] = summary_df["Variant_Richness"].fillna(0).astype(int)
    return summary_df


def add_functional_biallelic_calls(sample_df, include_loh_with_lof=True):
    sample_df = sample_df.copy()
    rows = []
    for _, row in sample_df.iterrows():
        het = split_genes(row.get("genes_HETDEL_str"))
        loh = split_genes(row.get("genes_LOH_str"))
        hom = split_genes(row.get("genes_HOMDEL_str"))
        lof_one = split_genes(row.get("LOF_OneAllele_Genes"))
        lof_reported_biallelic = split_genes(row.get("LOF_Biallelic_Genes"))
        lof_any = lof_one | lof_reported_biallelic
        functional = set(hom) | (het & lof_any)
        if include_loh_with_lof:
            functional |= loh & lof_any
        functional = sorted(functional)
        rows.append({
            "functional_biallelic_genes": ",".join(functional) if functional else "",
            "functional_biallelic_loss": int(bool(functional)),
            "functional_biallelic_loss_n_genes": len(functional),
        })
    return sample_df.join(pd.DataFrame(rows, index=sample_df.index))


def compute_oncocycle(row, loss_set, gain_set):
    functional_biallelic = split_genes(row.get("functional_biallelic_genes", ""))
    amps = split_genes(row.get("genes_amp_str", ""))
    gained = split_genes(row.get("genes_gained_str", ""))
    return int(bool(loss_set & functional_biallelic) or bool(gain_set & (amps | gained)))


def parse_ann_records(ann_value):
    records = []
    for record in str(ann_value or "").split(","):
        fields = record.split("|")
        while len(fields) < 18:
            fields.append("")
        records.append(fields)
    return records


def pick_best_effect_for_alt(records):
    best_record = None
    best_effect = None
    best_rank = 10 ** 9
    for record in records:
        for effect in (record[1] or "").split("&"):
            effect = effect.strip()
            rank = SEVERITY_RANK.get(effect, 10 ** 9)
            if rank < best_rank:
                best_record = record
                best_effect = effect
                best_rank = rank
    return best_record, best_effect


def classify_tmb_variant(effect, ref, alt):
    base = EFFECT_TO_CLASS.get(str(effect).strip())
    if base is None:
        return None
    if base == "Frame_Shift":
        if len(alt) > len(ref):
            return "Frame_Shift_Ins"
        if len(alt) < len(ref):
            return "Frame_Shift_Del"
        return "Frame_Shift_Ins"
    return base


def merge_intervals_0based(intervals):
    per_chr = defaultdict(list)
    for chrom, start0, end0 in intervals:
        if start0 < end0:
            per_chr[chrom].append((start0, end0))
    merged = {}
    for chrom, values in per_chr.items():
        values.sort()
        out = []
        for start0, end0 in values:
            if not out or start0 > out[-1][1]:
                out.append([start0, end0])
            else:
                out[-1][1] = max(out[-1][1], end0)
        merged[chrom] = [(start0, end0) for start0, end0 in out]
    return merged


def load_targets_padded_0based(path, pad=0):
    intervals = []
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip() or line.startswith("#"):
                continue
            parts = line.strip().split()
            if len(parts) < 3:
                continue
            chrom = parts[0]
            start0 = max(0, int(parts[1]) - pad)
            end0 = int(parts[2]) + pad
            intervals.append((chrom, start0, end0))
    return merge_intervals_0based(intervals)


def territory_bp_0based(merged):
    return sum(end0 - start0 for chrom in merged for start0, end0 in merged[chrom])


def overlaps_0based(chrom, start0, end0, merged):
    for target_start, target_end in merged.get(chrom, []):
        if target_end <= start0:
            continue
        if target_start >= end0:
            break
        if max(target_start, start0) < min(target_end, end0):
            return True
    return False


def clipped_target_overlaps(chrom, start0, end0, merged):
    overlaps = []
    for target_start, target_end in merged.get(chrom, []):
        if target_end <= start0:
            continue
        if target_start >= end0:
            break
        clip_start = max(target_start, start0)
        clip_end = min(target_end, end0)
        if clip_start < clip_end:
            overlaps.append((clip_start, clip_end))
    return overlaps


def merge_intervals(intervals):
    if not intervals:
        return []
    intervals = sorted(intervals)
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        prev_start, prev_end = merged[-1]
        if start <= prev_end:
            merged[-1] = (prev_start, max(prev_end, end))
        else:
            merged.append((start, end))
    return merged


def union_len(intervals):
    return sum(end - start for start, end in merge_intervals(intervals))


def load_sample_panel_map(path):
    mel_annotation = pd.read_excel(path, sheet_name="MFT_SINNBERG_QHPRG_MeasTabelle9")
    return (
        mel_annotation[["code", "system_name_short"]]
        .dropna(subset=["code"])
        .assign(
            code=lambda df: df["code"].astype(str).str.strip(),
            system_name_short=lambda df: df["system_name_short"].astype(str).str.strip(),
        )
        .drop_duplicates(subset="code", keep="first")
        .set_index("code")["system_name_short"]
        .to_dict()
    )


def compute_tmb(sample_ids, sample_panel_map):
    panel_targets_overlap = {panel: load_targets_padded_0based(path, pad=TMB_OVERLAP_PADDING) for panel, path in PANEL_BED_PATHS.items()}
    panel_targets_territory = {panel: load_targets_padded_0based(path, pad=0) for panel, path in PANEL_BED_PATHS.items()}
    raw_panel_territory_mb = {panel: territory_bp_0based(targets) / 1e6 for panel, targets in panel_targets_territory.items()}

    records = []
    for pattern in VCF_GLOB:
        for path in sorted(glob.glob(str(VCF_DIR / pattern), recursive=True)):
            sample_code = normalize_sample_id(sample_name_from_path(path))
            if sample_code not in sample_ids:
                continue
            panel_label = str(sample_panel_map.get(sample_code, "")).lower()
            panel_version = "v4" if "v4" in panel_label else "v5"
            overlap_targets = panel_targets_overlap[panel_version]
            denominator_mb = raw_panel_territory_mb[panel_version]
            nonsyn = 0

            with open_maybe_gzip(path) as handle:
                for line in handle:
                    if not line or line.startswith("#"):
                        continue
                    parts = line.rstrip("\n").split("\t")
                    if len(parts) < 8:
                        continue
                    chrom, pos, _id, ref, alts, qual, flt, info = parts[:8]
                    if flt != "PASS" or not looks_somatic(info, strict=STRICT_SOMATIC):
                        continue
                    info_dict = parse_info_dict(info)
                    ann_raw = info_dict.get("ANN")
                    if not ann_raw:
                        continue
                    start0 = int(pos) - 1
                    end0 = start0 + len(ref)
                    if not overlaps_0based(chrom, start0, end0, overlap_targets):
                        continue
                    ann_records = parse_ann_records(ann_raw)
                    for alt in alts.split(","):
                        records_for_alt = [rec for rec in ann_records if (rec[0] or "").split("/")[0] == alt] or ann_records
                        best_record, best_effect = pick_best_effect_for_alt(records_for_alt)
                        if not best_record or not best_effect:
                            continue
                        variant_class = classify_tmb_variant(best_effect, ref, alt)
                        if variant_class in NONSYN_CLASSES:
                            nonsyn += 1
            records.append({
                "Sample": sample_code,
                "TMB_per_Mb": nonsyn / denominator_mb if denominator_mb > 0 else np.nan,
                "Nonsynonymous_Count": nonsyn,
            })
    if not records:
        return pd.DataFrame(columns=["TMB_per_Mb", "Nonsynonymous_Count"])
    return pd.DataFrame.from_records(records).set_index("Sample").sort_index().groupby(level=0).median()


def compute_fga(sample_ids, sample_panel_map):
    panel_targets = {panel: load_targets_padded_0based(path, pad=FGA_INTERVAL_PADDING) for panel, path in PANEL_BED_PATHS.items()}
    panel_bp = {panel: territory_bp_0based(targets) for panel, targets in panel_targets.items()}

    records = []
    for path in iter_clincnv_paths(CLINCNV_ROOT):
        sample_code = sample_from_clincnv_path(path).split("-")[0]
        if sample_code not in sample_ids:
            continue
        panel_label = str(sample_panel_map.get(sample_code, "")).lower()
        panel_version = "v4" if "v4" in panel_label else "v5"
        merged_targets = panel_targets[panel_version]
        by_chr = defaultdict(list)

        for row in _read_clincnv_table(path):
            chrom = row.get("chr") or row.get("#chr") or row.get("chrom") or row.get("Chromosome")
            chrom = str(chrom)
            if chrom and not chrom.startswith("chr"):
                chrom = f"chr{chrom}"
            if chrom not in merged_targets:
                continue
            try:
                start = int(row.get("start", "0"))
                end = int(row.get("end", "0"))
            except Exception:
                continue
            if start >= end:
                continue
            if str(row.get("state", "")).upper() in {"AMP", "DEL"}:
                by_chr[chrom].extend(clipped_target_overlaps(chrom, start, end, merged_targets))

        altered_bp = sum(union_len(by_chr.get(chrom, [])) for chrom in merged_targets)
        records.append({
            "Sample": sample_code,
            "FGA_percent": 100.0 * altered_bp / panel_bp[panel_version] if panel_bp[panel_version] else np.nan,
        })

    if not records:
        return pd.DataFrame(columns=["FGA_percent"])
    return pd.DataFrame.from_records(records).set_index("Sample").sort_index().groupby(level=0).median()


def first_present_column(frame, candidates):
    for column in candidates:
        if column in frame.columns:
            return column
    return None


def normalize_yes_no(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    if text in {"yes", "y", "1", "true", "positive"}:
        return 1
    if text in {"no", "n", "0", "false", "negative"}:
        return 0
    return pd.NA


def collapse_stage(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().upper()
    if text in {"IA", "IB"}:
        return "I"
    if text in {"I", "IIA", "IIB", "IIC"}:
        return text
    return pd.NA


def derive_thickness_group(value):
    value = pd.to_numeric(value, errors="coerce")
    if pd.isna(value):
        return pd.NA
    if value < 2:
        return "below 2"
    if value <= 4:
        return "2 to 4"
    return "above 4"


def recode_histology(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    mapping = {
        "unknown cutaneous": "Other",
        "spindel cell": "Other",
        "Rare": "Other",
        "nevoid": "Other",
        "spitzoid": "Other",
        "verrucous": "Other",
        "desmoplastic": "Other",
        "polypoid": "Other",
    }
    return mapping.get(text, text)


def load_clinical_table(path):
    frame = pd.read_csv(path, sep="\t")
    missing = [column for column in REQUIRED_CLINICAL_COLUMNS if column not in frame.columns]
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")

    resolved = frame.copy()
    unresolved = []
    for target, candidates in CLINICAL_COLUMN_ALIASES.items():
        found = first_present_column(frame, candidates)
        if found is None:
            unresolved.append(target)
        else:
            resolved[target] = frame[found]
    if unresolved:
        raise ValueError(
            "The shared clinical TSV is missing the extra columns required for the Cox notebook: "
            + ", ".join(unresolved)
        )

    resolved["Sample"] = resolved["Sample"].astype(str).str.strip()
    if resolved["Sample"].duplicated().any():
        duplicates = sorted(resolved.loc[resolved["Sample"].duplicated(), "Sample"].unique())
        raise ValueError(f"Clinical metadata contains duplicated Sample IDs: {duplicates}")
    resolved = resolved.set_index("Sample", drop=False).sort_index()

    resolved["patient_id"] = resolved["patient_id"].astype(str).str.strip()
    resolved["relapse_any"] = pd.to_numeric(resolved["relapse_any"], errors="coerce").astype("Int64")
    resolved["relapse_120m"] = pd.to_numeric(resolved["relapse_120m"], errors="coerce").astype("Int64")
    resolved["rfs_time_months"] = pd.to_numeric(resolved["rfs_time_months"], errors="coerce")
    resolved["age_at_diagnosis_years"] = pd.to_numeric(resolved["age_at_diagnosis_years"], errors="coerce")
    resolved["sex"] = resolved["sex"].astype(str).str.strip().str.lower()
    resolved["ajcc8_stage"] = resolved["ajcc8_stage"].apply(collapse_stage)
    resolved["tumor_thickness_mm"] = pd.to_numeric(resolved["tumor_thickness_mm"], errors="coerce")
    resolved["tumor_thickness_group"] = resolved["tumor_thickness_group"].where(resolved["tumor_thickness_group"].notna(), resolved["tumor_thickness_mm"].map(derive_thickness_group))
    resolved["tumor_thickness_group"] = resolved["tumor_thickness_group"].astype("string").str.strip()
    resolved["ulceration"] = resolved["ulceration"].astype("string").str.strip()
    resolved["histology"] = resolved["histology"].apply(recode_histology)
    resolved["primary_site"] = resolved["primary_site"].astype("string").str.strip()
    resolved["sentinel"] = resolved["sentinel"].astype("string").str.strip().str.lower()
    resolved.loc[resolved["sentinel"].isin(["1", "true", "positive"]), "sentinel"] = "yes"
    resolved.loc[resolved["sentinel"].isin(["0", "false", "negative"]), "sentinel"] = "no"
    resolved["regression"] = resolved["regression"].astype("string").str.strip().fillna("Missing")
    resolved["nevus_association"] = resolved["nevus_association"].astype("string").str.strip().fillna("Missing")
    resolved["sentinel_binary"] = resolved["sentinel"].map(normalize_yes_no).astype("Int64")
    if int(resolved["sentinel_binary"].notna().sum()) == 0:
        raise ValueError("The sentinel column could not be normalized to yes/no values. Preformat it outside the repo before running this notebook.")
    resolved["ulceration_binary"] = resolved["ulceration"].map(normalize_yes_no).astype("Int64")
    resolved["rfs_full"] = resolved["rfs_time_months"]
    resolved["event_full"] = pd.to_numeric(resolved["relapse_any"], errors="coerce")
    resolved["rfs_120m"] = resolved["rfs_time_months"].clip(upper=120)
    resolved["event_120m"] = pd.to_numeric(resolved["relapse_120m"], errors="coerce")
    return resolved


def build_patient_level_dataframe():
    clinical_df = load_clinical_table(CLINICAL_METADATA_PATH)
    sample_ids = set(clinical_df.index)
    sample_panel_map = load_sample_panel_map(MEL_ANNOTATION_PATH)

    clincnv_summary = build_clincnv_summary(CLINCNV_ROOT)
    oncokb_summary = build_oncokb_summary(ONCOKB_DIR, VCF_DIR, include_samples=sample_ids)
    merged = add_functional_biallelic_calls(clincnv_summary.join(oncokb_summary, how="left"))

    patient_df = clinical_df.join(merged, how="inner")
    patient_df = patient_df.loc[~patient_df.index.duplicated(keep="first")].copy()
    if patient_df.empty:
        raise RuntimeError(
            "The Cox notebook found no overlap between the shared clinical TSV and the staged ClinCNV/OncoKB sample IDs. "
            "Check that Sample uses primary QHPRG tumor codes and that the staged outputs were built from the same cohort."
        )

    tmb_df = compute_tmb(set(patient_df.index), sample_panel_map)
    fga_df = compute_fga(set(patient_df.index), sample_panel_map)
    patient_df = patient_df.join(tmb_df, how="left")
    patient_df = patient_df.join(fga_df, how="left")

    for gene in LOSS_GENES_SIX:
        patient_df[f"{gene} loss"] = patient_df["functional_biallelic_genes"].map(lambda value, g=gene: int(g in split_genes(value)))
    for gene in GAIN_GENES_SIX:
        patient_df[f"{gene} gain"] = patient_df.apply(
            lambda row, g=gene: int(g in (split_genes(row.get("genes_amp_str", "")) | split_genes(row.get("genes_gained_str", "")))),
            axis=1,
        )

    patient_df["oncocycle_positive"] = patient_df.apply(
        lambda row: compute_oncocycle(row, set(LOSS_GENES_SIX), set(GAIN_GENES_SIX)),
        axis=1,
    ).astype(int)
    patient_df["oncocycle_features"] = patient_df.apply(
        lambda row: ";".join([feature for feature in ONCOCYCLE_FEATURES if int(row.get(feature, 0)) == 1]),
        axis=1,
    )

    oncogenic_sets = patient_df.get("Oncogenic_Genes", pd.Series("", index=patient_df.index)).map(split_genes)
    gene_lookup = {
        "TERT_oncogenic": "TERT",
        "BRAF_oncogenic": "BRAF",
        "NRAS_oncogenic": "NRAS",
        "NF1_oncogenic": "NF1",
        "KIT_oncogenic": "KIT",
        "CDKN2A_oncogenic": "CDKN2A",
        "TP53_oncogenic": "TP53",
        "RAC1_oncogenic": "RAC1",
        "MAP2K1_oncogenic": "MAP2K1",
        "LRP1B_oncogenic": "LRP1B",
        "PTPRT_oncogenic": "PTPRT",
        "PTPRD_oncogenic": "PTPRD",
        "ARID2_oncogenic": "ARID2",
        "PPP6C_oncogenic": "PPP6C",
    }
    for column, gene in gene_lookup.items():
        patient_df[column] = oncogenic_sets.map(lambda genes, g=gene: int(g in genes))
    patient_df["BRAF_NRAS_NF1_any"] = oncogenic_sets.map(lambda genes: int(bool({"BRAF", "NRAS", "NF1"} & genes)))

    tmb_cutoff = float(pd.to_numeric(patient_df["TMB_per_Mb"], errors="coerce").median())
    fga_cutoff = float(pd.to_numeric(patient_df["FGA_percent"], errors="coerce").median())
    patient_df["tmb_group"] = pd.to_numeric(patient_df["TMB_per_Mb"], errors="coerce").map(
        lambda value: pd.NA if pd.isna(value) else (f"Below {tmb_cutoff:.4f}" if value <= tmb_cutoff else f"Above {tmb_cutoff:.4f}")
    )
    patient_df["fga_group"] = pd.to_numeric(patient_df["FGA_percent"], errors="coerce").map(
        lambda value: pd.NA if pd.isna(value) else (f"Below {fga_cutoff:.4f}" if value <= fga_cutoff else f"Above {fga_cutoff:.4f}")
    )
    return patient_df, tmb_cutoff, fga_cutoff


def fit_numeric_cox(df, time_col, event_col, predictor, penalizer=0.01):
    work = pd.DataFrame({
        "time": pd.to_numeric(df[time_col], errors="coerce"),
        "event": pd.to_numeric(df[event_col], errors="coerce"),
        "x": pd.to_numeric(df[predictor], errors="coerce"),
    }).dropna()
    if work.empty or work["event"].sum() == 0 or work["x"].nunique() < 2:
        return None, work
    cph = CoxPHFitter(penalizer=penalizer)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cph.fit(work, duration_col="time", event_col="event")
    return cph, work


def fit_categorical_cox(df, time_col, event_col, predictor, levels, include_missing=False, penalizer=0.01):
    cat = df[predictor].copy()
    if include_missing:
        cat = cat.fillna("Missing")
    work = pd.DataFrame({
        "time": pd.to_numeric(df[time_col], errors="coerce"),
        "event": pd.to_numeric(df[event_col], errors="coerce"),
        "cat": cat,
    }).dropna(subset=["time", "event", "cat"])
    categories = list(levels)
    if include_missing and "Missing" not in categories and (work["cat"] == "Missing").any():
        categories.append("Missing")
    work = work[work["cat"].astype(str).isin([str(level) for level in categories])].copy()
    work["cat"] = pd.Categorical(work["cat"].astype(str), categories=[str(level) for level in categories], ordered=True)
    dummies = pd.get_dummies(work["cat"], prefix="cat", drop_first=True).astype(float)
    if dummies.empty:
        return None, work, dummies, np.nan
    design = pd.concat([work[["time", "event"]], dummies], axis=1)
    cph = CoxPHFitter(penalizer=penalizer)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cph.fit(design, duration_col="time", event_col="event")
    return cph, work, dummies, float(cph.log_likelihood_ratio_test().p_value)


def median_iqr(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    q1, median, q3 = values.quantile([0.25, 0.5, 0.75])
    return f"{median:.1f} [{q1:.1f}-{q3:.1f}]"


def count_and_pct(mask, denom):
    return f"{int(mask.sum())} ({(100 * mask.sum() / denom):.1f}%)" if denom else "0 (NA)"


def numeric_group_p_value(series, group):
    values = pd.to_numeric(series, errors="coerce")
    mask = values.notna() & group.notna()
    left = values[mask & (group == 0)]
    right = values[mask & (group == 1)]
    if len(left) == 0 or len(right) == 0:
        return np.nan
    return float(mannwhitneyu(left, right, alternative="two-sided").pvalue)


def categorical_group_p_value(series, group):
    table = pd.crosstab(series.fillna("Missing"), group)
    if table.shape[1] != 2 or table.shape[0] < 2:
        return np.nan
    if table.shape == (2, 2):
        return float(fisher_exact(table.to_numpy())[1])
    return float(chi2_contingency(table.to_numpy())[1])


def build_characteristics_table(patient_df):
    group = pd.to_numeric(patient_df["relapse_any"], errors="coerce")
    rows = [{
        "Characteristic": "Patients, n",
        "Overall": str(len(patient_df)),
        "No Relapse disease": str(int((group == 0).sum())),
        "Relapse": str(int((group == 1).sum())),
        "P value": "",
        "Univariable Cox HR [95% CI]": "",
        "Univariable Cox HR": np.nan,
        "Univariable Cox CI lower 95%": np.nan,
        "Univariable Cox CI upper 95%": np.nan,
        "Univariable Cox row p": np.nan,
        "Univariable Cox term p": np.nan,
        "Univariable Cox n": np.nan,
    }]

    def add_numeric(label, column_name):
        cph, work = fit_numeric_cox(patient_df, "rfs_full", "event_full", column_name, penalizer=0.01)
        summary = cph.summary.loc["x"] if cph is not None else None
        rows.append({
            "Characteristic": label,
            "Overall": median_iqr(patient_df[column_name]),
            "No Relapse disease": median_iqr(patient_df.loc[group == 0, column_name]),
            "Relapse": median_iqr(patient_df.loc[group == 1, column_name]),
            "P value": fmt_p_plain(numeric_group_p_value(patient_df[column_name], group)),
            "Univariable Cox HR [95% CI]": "" if summary is None else fmt_hr_ci(summary["exp(coef)"], summary["exp(coef) lower 95%"], summary["exp(coef) upper 95%"]),
            "Univariable Cox HR": np.nan if summary is None else float(summary["exp(coef)"]),
            "Univariable Cox CI lower 95%": np.nan if summary is None else float(summary["exp(coef) lower 95%"]),
            "Univariable Cox CI upper 95%": np.nan if summary is None else float(summary["exp(coef) upper 95%"]),
            "Univariable Cox row p": np.nan if summary is None else float(summary["p"]),
            "Univariable Cox term p": np.nan if summary is None else float(summary["p"]),
            "Univariable Cox n": len(work),
        })

    def add_categorical(label, column_name, levels, include_missing=False, show_levels=None, reference_level=None):
        series = patient_df[column_name]
        descriptive_p = categorical_group_p_value(series, group)
        cph, work, dummies, term_p = fit_categorical_cox(patient_df, "rfs_full", "event_full", column_name, levels, include_missing=include_missing, penalizer=0.01)
        if show_levels is None:
            show_levels = list(levels)
            if include_missing and series.isna().any() and "Missing" not in show_levels:
                show_levels.append("Missing")
        if reference_level is None:
            reference_level = show_levels[0]
        for idx, level in enumerate(show_levels):
            row = {
                "Characteristic": f"{label} = {level}",
                "Overall": count_and_pct(series.isna() if level == "Missing" else (series.astype(str) == str(level)), len(group)),
                "No Relapse disease": count_and_pct((series.isna() if level == "Missing" else (series.astype(str) == str(level))) & (group == 0), int((group == 0).sum())),
                "Relapse": count_and_pct((series.isna() if level == "Missing" else (series.astype(str) == str(level))) & (group == 1), int((group == 1).sum())),
                "P value": fmt_p_plain(descriptive_p) if idx == 0 else "",
                "Univariable Cox term p": term_p,
                "Univariable Cox n": len(work),
                "Univariable Cox row p": np.nan,
                "Univariable Cox HR": np.nan,
                "Univariable Cox CI lower 95%": np.nan,
                "Univariable Cox CI upper 95%": np.nan,
                "Univariable Cox HR [95% CI]": "Reference" if str(level) == str(reference_level) else "",
            }
            if str(level) == str(reference_level):
                row["Univariable Cox HR"] = 1.0
            else:
                col_name = f"cat_{level}"
                if cph is not None and col_name in cph.summary.index:
                    s = cph.summary.loc[col_name]
                    row["Univariable Cox HR [95% CI]"] = fmt_hr_ci(s["exp(coef)"], s["exp(coef) lower 95%"], s["exp(coef) upper 95%"])
                    row["Univariable Cox HR"] = float(s["exp(coef)"])
                    row["Univariable Cox CI lower 95%"] = float(s["exp(coef) lower 95%"])
                    row["Univariable Cox CI upper 95%"] = float(s["exp(coef) upper 95%"])
                    row["Univariable Cox row p"] = float(s["p"])
            rows.append(row)

    add_numeric("Age at diagnosis, median [IQR], years", "age_at_diagnosis_years")
    add_categorical("Sex", "sex", ["female", "male"], show_levels=["female", "male"], reference_level="female")
    add_categorical("AJCC8 stage", "ajcc8_stage", ["I", "IIA", "IIB", "IIC"], show_levels=["I", "IIA", "IIB", "IIC"], reference_level="I")
    add_numeric("Tumor thickness, median [IQR], mm", "tumor_thickness_mm")
    add_categorical("Tumor thickness group", "tumor_thickness_group", ["below 2", "2 to 4", "above 4"], show_levels=["below 2", "2 to 4", "above 4"], reference_level="below 2")
    add_categorical("Ulceration", "ulceration", ["No", "Yes"], show_levels=["No", "Yes"], reference_level="No")
    add_categorical("Histology", "histology", ["SSM", "NM", "ALM", "LMM", "Other"], show_levels=["SSM", "NM", "ALM", "LMM", "Other"], reference_level="SSM")
    add_categorical("Primary site", "primary_site", ["head and neck", "Upper extremities", "trunk", "lower extremities"], show_levels=["head and neck", "Upper extremities", "trunk", "lower extremities"], reference_level="head and neck")
    add_categorical("SLNB", "sentinel", ["no", "yes"], show_levels=["no", "yes"], reference_level="no")
    add_categorical("Regression", "regression", ["No", "Yes", "Missing"], include_missing=True, show_levels=["No", "Yes", "Missing"], reference_level="No")
    add_categorical("Nevus association", "nevus_association", ["No", "Yes", "Missing"], include_missing=True, show_levels=["No", "Yes", "Missing"], reference_level="No")
    return pd.DataFrame(rows)


def build_gene_univariable_table(patient_df):
    rows = []
    for feature in MAJOR_ONCOGENIC_GENE_FEATURES:
        work = pd.DataFrame({
            "time": pd.to_numeric(patient_df["rfs_full"], errors="coerce"),
            "event": pd.to_numeric(patient_df["event_full"], errors="coerce"),
            "feature": pd.to_numeric(patient_df[feature], errors="coerce"),
        }).dropna()
        row = {
            "outcome": "Full RFS",
            "feature": feature,
            "n": int(len(work)),
            "n_events": int(work["event"].sum()),
            "n_positive": int(work["feature"].sum()),
            "hazard_ratio": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "p_value": np.nan,
        }
        if work.empty or work["event"].sum() == 0 or work["feature"].sum() == 0 or int((work["feature"] == 0).sum()) == 0 or int(work.loc[work["feature"] == 0, "event"].sum()) == 0:
            rows.append(row)
            continue
        cph = CoxPHFitter(penalizer=0.01)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            cph.fit(work, duration_col="time", event_col="event")
        s = cph.summary.loc["feature"]
        row.update({
            "hazard_ratio": float(s["exp(coef)"]),
            "ci_lower": float(s["exp(coef) lower 95%"]),
            "ci_upper": float(s["exp(coef) upper 95%"]),
            "p_value": float(s["p"]),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def wald_test(cph, cols):
    beta = cph.params_.loc[cols].to_numpy(dtype=float)
    var = cph.variance_matrix_.loc[cols, cols].to_numpy(dtype=float)
    stat = float(beta.T @ np.linalg.inv(var) @ beta)
    return stat, len(cols), float(chi2.sf(stat, len(cols)))


def make_design(df, time_col, event_col, covariates, tmb_cutoff, fga_cutoff, subset_mask=None, include_oncocycle=True):
    if subset_mask is None:
        subset_mask = pd.Series(True, index=df.index)
    work = df.loc[subset_mask].copy()
    design = pd.DataFrame({
        "time": pd.to_numeric(work[time_col], errors="coerce"),
        "event": pd.to_numeric(work[event_col], errors="coerce"),
    }, index=work.index)
    terms = []

    if include_oncocycle:
        design["oncocycle_positive"] = pd.to_numeric(work["oncocycle_positive"], errors="coerce")
        terms.append({"name": "oncocycle", "label": "OncoCycle", "cols": ["oncocycle_positive"], "levels": ["Positive vs Negative"]})

    if "stage" in covariates:
        stage = pd.Categorical(work["ajcc8_stage"], categories=["I", "IIA", "IIB", "IIC"], ordered=True)
        for level in ["IIA", "IIB", "IIC"]:
            design[f"stage_{level}"] = (stage == level).astype(float)
        design.loc[pd.isna(stage), ["stage_IIA", "stage_IIB", "stage_IIC"]] = np.nan
        terms.append({"name": "stage", "label": "AJCC8 stage", "cols": ["stage_IIA", "stage_IIB", "stage_IIC"], "levels": ["IIA vs I", "IIB vs I", "IIC vs I"]})

    if "thickness" in covariates:
        cat = pd.Categorical(work["tumor_thickness_group"], categories=["below 2", "2 to 4", "above 4"], ordered=True)
        design["thickness_2_4"] = (cat == "2 to 4").astype(float)
        design["thickness_gt4"] = (cat == "above 4").astype(float)
        design.loc[pd.isna(cat), ["thickness_2_4", "thickness_gt4"]] = np.nan
        terms.append({"name": "thickness", "label": "Tumor thickness", "cols": ["thickness_2_4", "thickness_gt4"], "levels": ["2-4 mm vs <=2 mm", ">4 mm vs <=2 mm"]})

    if "ulceration" in covariates:
        design["ulceration_yes"] = pd.to_numeric(work["ulceration_binary"], errors="coerce")
        terms.append({"name": "ulceration", "label": "Ulceration", "cols": ["ulceration_yes"], "levels": ["Yes vs No"]})

    if "tmb" in covariates:
        below_label = f"Below {tmb_cutoff:.4f}"
        above_label = f"Above {tmb_cutoff:.4f}"
        design["tmb_below"] = (work["tmb_group"] == below_label).astype(float)
        design.loc[work["tmb_group"].isna(), "tmb_below"] = np.nan
        terms.append({"name": "tmb", "label": f"TMB below {tmb_cutoff:.4f}", "cols": ["tmb_below"], "levels": [f"Below {tmb_cutoff:.4f} vs Above {tmb_cutoff:.4f}"]})

    if "fga" in covariates:
        design["fga_below"] = (work["fga_group"] == f"Below {fga_cutoff:.4f}").astype(float)
        design.loc[work["fga_group"].isna(), "fga_below"] = np.nan
        terms.append({"name": "fga", "label": f"FGA below {tmb_cutoff * 0 + fga_cutoff:.4f}%", "cols": ["fga_below"], "levels": [f"Below {fga_cutoff:.4f}% vs Above {fga_cutoff:.4f}%"]})

    model_cols = ["time", "event"] + [col for term in terms for col in term["cols"]]
    model_df = design[model_cols].dropna().copy()
    if model_df.empty or model_df["event"].sum() == 0:
        return None, model_df, terms
    cph = CoxPHFitter(penalizer=0.01)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cph.fit(model_df, duration_col="time", event_col="event")
    return cph, model_df, terms


def term_rows(cph, model_df, terms, cohort_label):
    rows = []
    summary = cph.summary
    for term in terms:
        if len(term["cols"]) > 1:
            stat, df_value, p_value = wald_test(cph, term["cols"])
            rows.append({
                "Variable": term["label"],
                "Level": "",
                "HR [95% CI]": "",
                "p value": p_value,
                "term p value": p_value,
                "n": len(model_df),
                "cohort": cohort_label,
            })
        for level, col in zip(term["levels"], term["cols"]):
            s = summary.loc[col]
            rows.append({
                "Variable": term["label"] if len(term["cols"]) == 1 else "",
                "Level": level,
                "HR [95% CI]": fmt_hr_ci(s["exp(coef)"], s["exp(coef) lower 95%"], s["exp(coef) upper 95%"]),
                "p value": float(s["p"]),
                "term p value": float(s["p"]),
                "n": len(model_df),
                "cohort": cohort_label,
            })
    return pd.DataFrame(rows)


def add_wide_table(df, time_col, event_col, covariates, tmb_cutoff, fga_cutoff):
    full_cph, full_model_df, full_terms = make_design(df, time_col, event_col, covariates, tmb_cutoff, fga_cutoff)
    sent_mask = df["sentinel_binary"].eq(1)
    sent_cph, sent_model_df, sent_terms = make_design(df, time_col, event_col, covariates, tmb_cutoff, fga_cutoff, subset_mask=sent_mask)

    full_rows = term_rows(full_cph, full_model_df, full_terms, "Entire cohort").rename(columns={
        "HR [95% CI]": "Entire cohort HR [95% CI]",
        "p value": "Entire cohort p value",
        "n": "Entire cohort n",
    })
    sent_rows = term_rows(sent_cph, sent_model_df, sent_terms, "Sentinel subset").rename(columns={
        "HR [95% CI]": "Sentinel subset HR [95% CI]",
        "p value": "Sentinel subset p value",
        "n": "Sentinel subset n",
    })

    merged = full_rows[["Variable", "Level", "Entire cohort HR [95% CI]", "Entire cohort p value", "Entire cohort n"]].merge(
        sent_rows[["Variable", "Level", "Sentinel subset HR [95% CI]", "Sentinel subset p value", "Sentinel subset n"]],
        on=["Variable", "Level"],
        how="outer",
    )
    return merged, full_cph, full_model_df, full_terms, sent_cph, sent_model_df, sent_terms


def build_stage_oncocycle_comparison(df):
    full_cph, full_model_df, _ = make_design(df, "rfs_full", "event_full", ["stage"], np.nanmedian(df["TMB_per_Mb"]), np.nanmedian(df["FGA_percent"]), include_oncocycle=True)
    censored_cph, censored_model_df, _ = make_design(df, "rfs_120m", "event_120m", ["stage"], np.nanmedian(df["TMB_per_Mb"]), np.nanmedian(df["FGA_percent"]), include_oncocycle=True)
    rows = []
    for label, cph in [("Full RFS", full_cph), ("120-month RFS", censored_cph)]:
        summary = cph.summary
        stat, df_value, p_stage = wald_test(cph, ["stage_IIA", "stage_IIB", "stage_IIC"])
        mapping = {
            "Stage (overall)": (np.nan, np.nan, np.nan, p_stage),
            "Stage IIA vs I": (summary.loc["stage_IIA", "exp(coef)"], summary.loc["stage_IIA", "exp(coef) lower 95%"], summary.loc["stage_IIA", "exp(coef) upper 95%"], summary.loc["stage_IIA", "p"]),
            "Stage IIB vs I": (summary.loc["stage_IIB", "exp(coef)"], summary.loc["stage_IIB", "exp(coef) lower 95%"], summary.loc["stage_IIB", "exp(coef) upper 95%"], summary.loc["stage_IIB", "p"]),
            "Stage IIC vs I": (summary.loc["stage_IIC", "exp(coef)"], summary.loc["stage_IIC", "exp(coef) lower 95%"], summary.loc["stage_IIC", "exp(coef) upper 95%"], summary.loc["stage_IIC", "p"]),
            "OncoCycle ": (summary.loc["oncocycle_positive", "exp(coef)"], summary.loc["oncocycle_positive", "exp(coef) lower 95%"], summary.loc["oncocycle_positive", "exp(coef) upper 95%"], summary.loc["oncocycle_positive", "p"]),
        }
        rows.append((label, mapping))
    order = ["Stage (overall)", "Stage IIA vs I", "Stage IIB vs I", "Stage IIC vs I", "OncoCycle "]
    out_rows = []
    for variable in order:
        full_vals = rows[0][1][variable]
        cen_vals = rows[1][1][variable]
        out_rows.append({
            "Variable": variable,
            "Full RFS HR [95% CI]": "" if pd.isna(full_vals[0]) else fmt_hr_ci(full_vals[0], full_vals[1], full_vals[2], digits=3),
            "Full RFS p": float(full_vals[3]),
            "120-month RFS HR [95% CI]": "" if pd.isna(cen_vals[0]) else fmt_hr_ci(cen_vals[0], cen_vals[1], cen_vals[2], digits=3),
            "120-month RFS p": float(cen_vals[3]),
        })
    return pd.DataFrame(out_rows)


def build_forest_frame_from_characteristics(table):
    rows = []
    order = 0
    for _, row in table.iterrows():
        hr = row["Univariable Cox HR"]
        low = row["Univariable Cox CI lower 95%"]
        high = row["Univariable Cox CI upper 95%"]
        p_value = row["Univariable Cox row p"]
        if pd.isna(hr) or pd.isna(low) or pd.isna(high) or float(hr) == 1.0 and row["Characteristic"].endswith("= I"):
            continue
        label = row["Characteristic"]
        if label == "Age at diagnosis, median [IQR], years":
            label = "Age at diagnosis: Per 1 year increase"
        elif label.startswith("Sex = "):
            level = label.split("= ", 1)[1]
            label = f"Sex: {level} vs female"
            if level == "female":
                continue
        elif label.startswith("AJCC8 stage = "):
            level = label.split("= ", 1)[1]
            if level == "I":
                continue
            label = f"AJCC8 stage: {level} vs I"
        elif label.startswith("Tumor thickness group = "):
            level = label.split("= ", 1)[1]
            if level == "below 2":
                continue
            label = f"Tumor thickness group: {level} vs below 2"
        elif label.startswith("Ulceration = "):
            level = label.split("= ", 1)[1]
            if level == "No":
                continue
            label = f"Ulceration: {level} vs No"
        elif label.startswith("Histology = "):
            level = label.split("= ", 1)[1]
            if level == "SSM":
                continue
            label = f"Histology: {level} vs SSM"
        elif label.startswith("Primary site = "):
            level = label.split("= ", 1)[1]
            if level == "head and neck":
                continue
            label = f"Primary site: {level} vs head and neck"
        elif label.startswith("SLNB = "):
            level = label.split("= ", 1)[1]
            if level == "no":
                continue
            label = f"SLNB: {level} vs no"
        elif label.startswith("Regression = "):
            level = label.split("= ", 1)[1]
            if level == "No":
                continue
            label = f"Regression: {level} vs No"
        elif label.startswith("Nevus association = "):
            level = label.split("= ", 1)[1]
            if level == "No":
                continue
            label = f"Nevus association: {level} vs No"
        rows.append({
            "Label": label,
            "HR": float(hr),
            "CI lower": float(low),
            "CI upper": float(high),
            "p": float(p_value),
            "order": order,
        })
        order += 1
    return pd.DataFrame(rows)


def build_forest_frame_from_gene_table(table):
    labels = {
        "BRAF_NRAS_NF1_any": "BRAF / NRAS / NF1 any",
        "TERT_oncogenic": "TERT",
        "BRAF_oncogenic": "BRAF",
        "NRAS_oncogenic": "NRAS",
        "NF1_oncogenic": "NF1",
        "KIT_oncogenic": "KIT",
        "CDKN2A_oncogenic": "CDKN2A",
        "TP53_oncogenic": "TP53",
        "RAC1_oncogenic": "RAC1",
        "MAP2K1_oncogenic": "MAP2K1",
        "LRP1B_oncogenic": "LRP1B",
        "PTPRT_oncogenic": "PTPRT",
        "PTPRD_oncogenic": "PTPRD",
        "ARID2_oncogenic": "ARID2",
        "PPP6C_oncogenic": "PPP6C",
    }
    rows = []
    for order, row in enumerate(table.itertuples(index=False)):
        if pd.isna(row.hazard_ratio):
            continue
        label = f"{labels.get(row.feature, row.feature)} (n={int(row.n_positive)})"
        rows.append({
            "Label": label,
            "HR": float(row.hazard_ratio),
            "CI lower": float(row.ci_lower),
            "CI upper": float(row.ci_upper),
            "p": float(row.p_value),
            "order": order,
        })
    return pd.DataFrame(rows)


def build_forest_frame_from_multivariable(table):
    rows = []
    order = 0
    for _, row in table.iterrows():
        if pd.isna(row["Entire cohort HR [95% CI]"]) or row["Level"] == "":
            continue
        hr_part, ci_part = row["Entire cohort HR [95% CI]"].split(" [", 1)
        low_part, high_part = ci_part.rstrip("]").split("-", 1)
        variable = row["Variable"] if row["Variable"] else "AJCC8 stage"
        label = row["Level"]
        if variable == "OncoCycle":
            label = "OncoCycle: Positive vs Negative"
        elif variable == "AJCC8 stage":
            label = f"AJCC8 stage: {label}"
        elif variable == "Tumor thickness":
            label = f"Tumor thickness: {label}"
        elif variable == "Ulceration":
            label = f"Ulceration: {label}"
        elif variable.startswith("TMB below "):
            label = "TMB: Below median vs above"
        elif variable.startswith("FGA below "):
            label = "FGA: Below median vs above"
        rows.append({
            "Label": label,
            "HR": float(hr_part),
            "CI lower": float(low_part),
            "CI upper": float(high_part),
            "p": float(row["Entire cohort p value"]),
            "order": order,
        })
        order += 1
    return pd.DataFrame(rows)


def draw_forest(ax, plot_df, title, color="#b35c1e"):
    plot_df = plot_df.sort_values("order").reset_index(drop=True)
    y_pos = np.arange(len(plot_df))[::-1]
    ax.errorbar(
        plot_df["HR"],
        y_pos,
        xerr=[plot_df["HR"] - plot_df["CI lower"], plot_df["CI upper"] - plot_df["HR"]],
        fmt="o",
        color=color,
        ecolor=color,
        elinewidth=2,
        capsize=4,
        markersize=6,
    )
    ax.axvline(1.0, color="black", linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_xticks([0.125, 0.25, 0.5, 1, 2, 4, 8, 16])
    ax.get_xaxis().set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:g}"))
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_df["Label"])
    ax.set_xlabel("Hazard ratio (log scale)")
    ax.set_title(title, loc="left", fontsize=12, fontweight="bold")
    ax.grid(axis="x", alpha=0.25)
    for x_value, y_value, p_value in zip(plot_df["HR"], y_pos, plot_df["p"]):
        if p_value < 0.05:
            ax.text(x_value * 1.08, y_value, "*", ha="left", va="center", fontsize=13, color=color, fontweight="bold")
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)


required_inputs = pd.DataFrame(
    [
        ("Clinical Metadata", CLINICAL_METADATA_PATH, False, "external metadata", "shared external clinical TSV with cleaned survival and clinicopathologic columns"),
        ("filtered annotated VCFs", VCF_DIR / "**/filtered.*.annotated.vcf*", True, "downstream output", "TMB reconstruction and PASS-backed OncoKB matching"),
        ("combined ClinCNV somatic outputs", CLINCNV_ROOT / "somatic/*/Annotated_ClinCNV_*.txt", True, "downstream of repo scripts", "six-gene OncoCycle CNV state summaries and FGA"),
        ("OncoKB tables", ONCOKB_DIR / "*.oncokb.tsv", True, "downstream output", "oncogenic gene flags and functional biallelic-loss support"),
        ("MEL annotation workbook", MEL_ANNOTATION_PATH, False, "downstream metadata", "map each sample to the correct v4 or v5 panel"),
        ("v4 target BED", PANEL_BED_PATHS["v4"], False, "downstream output", "panel-specific target territory for TMB and FGA"),
        ("v5 target BED", PANEL_BED_PATHS["v5"], False, "downstream output", "panel-specific target territory for TMB and FGA"),
    ],
    columns=["input_family", "location", "is_glob", "category", "used_for"],
)
required_inputs["matches"] = required_inputs.apply(lambda row: glob_count(row["location"]) if row["is_glob"] else path_count(row["location"]), axis=1)
required_inputs["present"] = required_inputs["matches"] > 0

display(Markdown("## Required staged inputs"))
display(required_inputs[["input_family", "category", "location", "used_for", "matches", "present"]])
missing_inputs = required_inputs.loc[~required_inputs["present"]].copy()
if not missing_inputs.empty:
    raise FileNotFoundError(
        "Missing staged inputs for Cox notebook:\n" + missing_inputs[["input_family", "location"]].to_string(index=False)
    )

clinical_contract = pd.DataFrame(
    [
        ("Sample", "QHPRG primary tumor code used as the join key"),
        ("patient_id", "paired FO patient identifier"),
        ("relapse_any", "full-duration RFS event flag used for the main Cox models"),
        ("relapse_120m", "administratively censored 120-month RFS event flag"),
        ("rfs_time_months", "full RFS time in months before 120-month clipping"),
        ("age_at_diagnosis_years or age_years", "numeric age at diagnosis"),
        ("sex", "female or male"),
        ("ajcc8_stage or stage_cat", "IA/IB/IIA/IIB/IIC; IA and IB collapse to I in the models"),
        ("tumor_thickness_mm", "numeric thickness in mm"),
        ("tumor_thickness_group", "below 2 / 2 to 4 / above 4"),
        ("ulceration", "No or Yes"),
        ("histology or histology_cat", "SSM / NM / ALM / LMM / Other"),
        ("primary_site", "head and neck / Upper extremities / trunk / lower extremities"),
        ("sentinel", "no or yes"),
        ("regression", "No / Yes / Missing"),
        ("nevus_association", "No / Yes / Missing"),
    ],
    columns=["column", "meaning"],
)
display(Markdown("## External clinical file contract"))
display(clinical_contract)

patient_df, tmb_cutoff, fga_cutoff = build_patient_level_dataframe()
summary_df = pd.DataFrame(
    [
        ("Primary cohort rows", len(patient_df)),
        ("Full-RFS events", int(pd.to_numeric(patient_df["event_full"], errors="coerce").sum())),
        ("120-month RFS events", int(pd.to_numeric(patient_df["event_120m"], errors="coerce").sum())),
        ("Sentinel subset rows", int(patient_df["sentinel_binary"].eq(1).sum())),
        ("OncoCycle positive", int(patient_df["oncocycle_positive"].sum())),
        ("Median TMB cutoff (/Mb)", round(tmb_cutoff, 4)),
        ("Median FGA cutoff (%)", round(fga_cutoff, 4)),
    ],
    columns=["metric", "value"],
)
display(Markdown("## Cohort summary"))
display(summary_df)

characteristics_table = build_characteristics_table(patient_df)
display(Markdown("## Table 1: clinical characteristics with descriptive P values and univariable Cox columns"))
display(characteristics_table)

gene_table = build_gene_univariable_table(patient_df)
display(Markdown("## Table 2: univariable full-RFS Cox models for major oncogenic mutation features"))
display(gene_table)

stage_tmb_fga_table, *_ = add_wide_table(patient_df, "rfs_full", "event_full", ["stage", "tmb", "fga"], tmb_cutoff, fga_cutoff)
display(Markdown("## Table 3: OncoCycle + AJCC8 stage + TMB + FGA multivariable Cox table"))
display(stage_tmb_fga_table)

stage_oncocycle_120m_table = build_stage_oncocycle_comparison(patient_df)
display(Markdown("## Table 4: stage + OncoCycle full-RFS versus 120-month-RFS comparison"))
display(stage_oncocycle_120m_table)

full_multivariable_table, *_ = add_wide_table(patient_df, "rfs_full", "event_full", ["stage", "thickness", "ulceration", "tmb", "fga"], tmb_cutoff, fga_cutoff)
display(Markdown("## Table 5: full multivariable RFS Cox table with OncoCycle, stage, thickness, ulceration, TMB, and FGA"))
display(full_multivariable_table)

clinical_plot_df = build_forest_frame_from_characteristics(characteristics_table)
gene_plot_df = build_forest_frame_from_gene_table(gene_table)
multivariable_plot_df = build_forest_frame_from_multivariable(full_multivariable_table)

fig, axes = plt.subplots(1, 3, figsize=(20, 11), gridspec_kw={"width_ratios": [1.2, 1.0, 1.0]})
draw_forest(axes[0], clinical_plot_df, "Clinical univariable Cox model")
draw_forest(axes[1], gene_plot_df, "Major oncogenic-gene univariable Cox models")
draw_forest(axes[2], multivariable_plot_df, "Full multivariable RFS Cox model")
fig.tight_layout()
plt.show()
